In [1]:
import pandas as pd
import torch
from torch import nn
import jieba
from gensim.models import KeyedVectors, Word2Vec

In [2]:
df = pd.read_csv("./data/online_shopping_10_cats.csv")

In [3]:
df.dropna(inplace=True)
print(df.isnull().sum())

cat       0
label     0
review    0
dtype: int64


In [4]:
sentence = [[token for token in jieba.lcut(line) if token.strip() != ""] for line in df["review"]]

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\YueLi\AppData\Local\Temp\jieba.cache
Loading model cost 0.281 seconds.
Prefix dict has been built successfully.


In [ ]:
# 模型训练
model = Word2Vec(
    sentences=sentence,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    workers=4
)

In [ ]:
# 保存模型
model.wv.save_word2vec_format("./data/word2vec.kv")

In [10]:
# 加载模型
kv_model = KeyedVectors.load_word2vec_format("./data/word2vec.kv")

In [21]:
text = "我喜欢乘坐宇宙飞船"
tokens = jieba.lcut(text)
print(tokens)

['我', '喜欢', '乘坐', '宇宙飞船']


In [24]:
print(kv_model.index_to_key)

['，', '的', '。', '了', '！', '很', '是', '我', '也', '好', '不', ',', '都', '就', '买', '不错', '还', '在', '有', '没有', '酒店', '用', '.', '京东', '和', '说', '房间', '给', '可以', '、', '这', '就是', '到', '非常', '一个', '感觉', '还是', '？', '没', '这个', '服务', '质量', '人', '比较', '上', '苹果', '要', '看', '去', '喜欢', '东西', '太', '不是', '又', '小', '但', '我们', '大', '价格', '让', '什么', '但是', '吧', '你', '差', '住', '而且', '多', '个', '…', '知道', '自己', '再', '才', '满意', '会', '啊', '有点', '：', '对', '比', '不好', '挺', '问题', '快递', '收到', '能', '吃', '手机', '2', '裤子', '来', '时候', '一般', '快', '还有', '一样', '?', '入住', '方便', '后', '真的', '这样', '一直', '~', '包装', '；', '蒙牛', '!', '物流', '着', '速度', '过', '以后', '不过', '3', '舒服', '客服', '特别', '一次', '不能', '穿', '*', '很多', '他', '味道', '送', '1', '）', '现在', '一点', '因为', '前台', '购买', '下次', '屏幕', '觉得', '已经', '早餐', '差评', '（', '想', '本书', '中', '时', '不会', '垃圾', '月', '第一次', '里', '跟', '把', '这次', '这么', '值得', '支持', '如果', '被', '效果', '呢', '得', '4', '便宜', '应该', '“', '很快', '”', '最', '起来', '发现', '一', '点', '等', '所以', '高', '希望', '大家', '设施', '失望', '怎么', '不要', '一下'

In [23]:
# 定义UNK
unk_token = '<UNK>'

In [27]:
# 将UNK添加到词表中
id2word = [unk_token] + kv_model.index_to_key
word2id = {token: index for index, token in enumerate(id2word)}
print(word2id)

{'<UNK>': 0, '，': 1, '的': 2, '。': 3, '了': 4, '！': 5, '很': 6, '是': 7, '我': 8, '也': 9, '好': 10, '不': 11, ',': 12, '都': 13, '就': 14, '买': 15, '不错': 16, '还': 17, '在': 18, '有': 19, '没有': 20, '酒店': 21, '用': 22, '.': 23, '京东': 24, '和': 25, '说': 26, '房间': 27, '给': 28, '可以': 29, '、': 30, '这': 31, '就是': 32, '到': 33, '非常': 34, '一个': 35, '感觉': 36, '还是': 37, '？': 38, '没': 39, '这个': 40, '服务': 41, '质量': 42, '人': 43, '比较': 44, '上': 45, '苹果': 46, '要': 47, '看': 48, '去': 49, '喜欢': 50, '东西': 51, '太': 52, '不是': 53, '又': 54, '小': 55, '但': 56, '我们': 57, '大': 58, '价格': 59, '让': 60, '什么': 61, '但是': 62, '吧': 63, '你': 64, '差': 65, '住': 66, '而且': 67, '多': 68, '个': 69, '…': 70, '知道': 71, '自己': 72, '再': 73, '才': 74, '满意': 75, '会': 76, '啊': 77, '有点': 78, '：': 79, '对': 80, '比': 81, '不好': 82, '挺': 83, '问题': 84, '快递': 85, '收到': 86, '能': 87, '吃': 88, '手机': 89, '2': 90, '裤子': 91, '来': 92, '时候': 93, '一般': 94, '快': 95, '还有': 96, '一样': 97, '?': 98, '入住': 99, '方便': 100, '后': 101, '真的': 102, '这样': 103, '一直': 104, '~': 105, '包

In [30]:
ids = [word2id.get(token, word2id[unk_token]) for token in tokens]
print(ids)

[8, 50, 5820, 0]


In [31]:
input = torch.tensor(ids)

In [39]:
# 重新定义词嵌入层
embedding_matrix = torch.cat(
    (torch.zeros(1, 100),
    torch.tensor(kv_model.vectors))
)
embedding = nn.Embedding.from_pretrained(
    embeddings=embedding_matrix,
    freeze=False
)

In [42]:
print(embedding.weight)
print(embedding.weight.shape)

Parameter containing:
tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.1748,  0.1022, -0.0047,  ..., -0.1064, -0.1048,  0.0723],
        [-0.2262,  0.0503, -0.0366,  ...,  0.0106, -0.1670,  0.0004],
        ...,
        [ 0.0087,  0.0556,  0.0509,  ..., -0.0729,  0.0188, -0.1312],
        [-0.0059,  0.0715,  0.0412,  ..., -0.0784,  0.0215, -0.1495],
        [ 0.0111,  0.0819,  0.0509,  ..., -0.1198,  0.0013, -0.1470]],
       requires_grad=True)
torch.Size([34577, 100])


In [41]:
output = embedding(input)
print(output)

tensor([[ 6.6358e-02,  6.9939e-02,  1.7209e-01, -1.6862e-01, -1.7334e-01,
         -4.3412e-01,  6.0102e-02,  2.8819e-01, -2.7802e-02, -1.9392e-01,
          3.5228e-01, -1.6119e-01,  1.5688e-01, -1.5995e-01, -1.9324e-01,
          6.6740e-02,  1.7183e-01, -3.5186e-02,  1.3246e-01, -5.6466e-01,
          1.7600e-01,  8.9360e-02,  6.3621e-02, -2.6732e-01, -2.2252e-01,
          1.5645e-01,  1.0471e-01, -1.5575e-02,  4.1271e-02, -3.6166e-01,
          3.4690e-01, -3.4806e-02,  1.0663e-01, -3.0640e-02, -2.5780e-02,
         -1.7041e-01,  2.3118e-01, -2.6122e-01, -1.0619e-01, -3.4899e-01,
          1.0784e-01, -3.3289e-01,  2.0112e-01,  2.9422e-01,  2.7341e-01,
         -1.0752e-01, -8.8524e-02, -3.9933e-02,  3.4263e-01,  3.2132e-01,
         -6.1960e-01, -1.4521e-01,  2.6281e-01,  2.1476e-01, -5.3045e-01,
          2.4358e-01,  5.1384e-02,  1.7409e-01, -3.0054e-01,  3.6195e-01,
          4.1071e-01, -1.7855e-01, -3.0999e-01, -4.9846e-02, -4.0445e-02,
          4.0201e-01,  3.5603e-02,  4.